# HRAI API demo notebook
použij `uvicorn main:app` (`--port <volný port>`)

In [ ]:
!pip install --upgrade pip
!pip install -r requirements.txt

In [1]:
import os, json, random, textwrap
from pathlib import Path

from pos_extraction import text_to_ngrams
from config import config
from dotenv import load_dotenv

In [ ]:
# pro rychlejší fungování encoderu doporučuji nastavit HF_TOKEN:
os.environ['HF_TOKEN'] = ''

In [2]:
load_dotenv()

True

## definice funkcí

In [ ]:
from query import extract_from_resume, query_type
from suggestions import match_occupations, get_domain_skills
from parse_doc import extract_text

resume_path = Path(os.path.join(config.data_dir, 'cs_resumes.json'))

def pick_resume():
    resumes = json.loads(resume_path.read_text(encoding='utf-8'))
    resume = random.choice(resumes)
    return resume.strip()


def documentinput_job_specific(filepath: str, target_job: str = None):
    """
    in: PDF/DOCX/ODT + target job
    out: top suggestion + target suggestion
    """
    with open(filepath, 'rb') as f: b = f.read()
    filename = os.path.basename(filepath)
    
    text = extract_text(filename, b)
    entities = extract_from_resume(text)
    skills = [e for e in entities if e.entity_type == 'skill']
    occupations = [e for e in entities if e.entity_type == 'occupation']
    suggestions = match_occupations(skills, occupations, target_job)
    
    target_suggestion = None
    if target_job:
        target_suggestions = match_occupations(skills, None, target_job)
        target_suggestion = target_suggestions[0] if target_suggestions else None
    
    return {
        'top_suggestion': suggestions[0],
        'target_suggestion': target_suggestion,
        'extracted_skills': skills,
        'extracted_occupations': occupations
    }


def documentinput_job_specific_isco(filepath: str):
    """
    in: PDF/DOCX/ODT + target job
    out: top suggestion for each top level isco group
    """
    with open(filepath, 'rb') as f: file_bytes = f.read()
    filename = os.path.basename(filepath)
    
    text = extract_text(filename, file_bytes)
    entities = extract_from_resume(text)
    skills = [e for e in entities if e.entity_type == 'skill']
    occupations = [e for e in entities if e.entity_type == 'occupation']
    suggestions = match_occupations(skills, occupations, None)
    domains = get_domain_skills(suggestions)

    return domains


def textinput_match(skills_str: str):
    """
    in: skills string
    out: match ordered suggestions
    """
    skill_labels = [s.strip() for s in skills_str.split(",")]
    skill_entities = query_type(skill_labels, 'skill')
    occupation_entities = query_type(skill_labels, 'occupation')
    
    suggestions = match_occupations(skill_entities, occupation_entities, None)
    return suggestions


def textinput_job_specific_match(skills_str: str, target_job: str):
    """
    in: skills string
    out: top matched occupation, target job match
    """
    skill_labels = [s.strip() for s in skills_str.split(",")]
    skill_entities = query_type(skill_labels, 'skill')
    occupation_entities = query_type(skill_labels, 'occupation')
    
    top_suggestions = match_occupations(skills=skill_entities, occupations=occupation_entities)
    target_suggestions = match_occupations(skills=skill_entities, occupations=None, target_job=target_job)
    return {
        'top_suggestion': top_suggestions[0],
        'target_suggestion': target_suggestions[0]
    }


def text_search(query_text: str, entity_type: str = None, k: int = 5):
    """
    in: entity string
    out: k entity results
    """
    label = entity_type
    results = query_type([query_text], label, search_k=k)
    return results


def print_suggestion(suggestion, title="Suggestion"):
    if suggestion is None:
        print(f"{title}: None")
        return
    print(f"\n{title}:")
    print(f"  Povolání: {suggestion.occupation.label}")
    print(f"  ISCO kód: {suggestion.occupation.code}")
    print(f"  Match score: {suggestion.match_score:.2%}")
    print(f"  Chybějící dovednosti: {len(suggestion.missing_skills)}")
    if suggestion.missing_skills[:3]:
        print(f"  Příklady chybějících:")
        for skill in suggestion.missing_skills[:3]:
            print(f"    - {skill.label} ({skill.relation_type})")


def print_entity(entity):
    """Helper to print an entity result"""
    print(f"  [{entity.entity_type}] {entity.label} (score: {entity.cosine_score:.3f})")


## Ukázky

In [5]:
text = pick_resume()
print(textwrap.shorten(text, width=600, placeholder='...'))

HR koordinátorka, která přináší 10 let práce s rozvojem efektivních mzdových a benefitních procesů v rámci firemního nastavení lidských zdrojů. Umí se učit nové průmyslové zákony a normy a také začleňovat příslušné osvědčené postupy do nového plánování a koordinace. Dovednosti Benefity a koordinace mezd Vynikající interpersonální dovednosti Detailně orientované prověrky Time management Výstupní pohovory Nábor a udržení zaměstnanců ADP Plynně hovořící anglicky Lawson Pracovní historie Sr. Dovolená administrativního specialisty 06/ k Současný Název společnosti – Rozsáhlé znalosti plánů...


#### vytvoření frází

In [6]:
ngrams = text_to_ngrams(text)
for phrase in ngrams:
    print(phrase)

každodenního provozu
zákazníků
metrik a opatření
společnosti zodpovídá
10 název společnosti
nesrovnalosti
střediska služeb
cornell university listopad
zajišťovali soulad
společnosti spravoval
výhody
university lidské zdroje
zdrojů
přesně do účetnictví
postupech dovolené
05 název
rozhodnutí
imigrační proces
politiky procesy
hr
produkty programy
společnost
rozhodnutí a přijímá
dodavatelů vize
komunikaci se zaměstnancem
název společnosti spravoval
procesů
výhod objasnit
strategii poskytoval
zaměstnancích změny stavu
orientaci pro zaměstnance
manažery a příležitostně
dovedností
university listopad přidružení
rozsáhlé znalosti
předpisy
postižením a dovolenou
ada adp
spoření plán
programů pro osoby
snížení nároků
řídil všechny aspekty
zdravotního spoření plán
nábor zápis
řešení problémů
lidských zdrojů zaznamenáváním
roi metrik
zaměstnanci na zavádění
programy a služby
zúčastněným stranám řídili
pojištění lawson řízení
zrakové dlouhodobé invalidity
doby odchodu
nových zaměstnanců přesunů
vyř

#### získání klíčových slov a oborů (pomocí pick_resume)

In [7]:
results = extract_from_resume(text)
for r in results:
    print(
        f'{r.label}\n'
        f'skore: {r.cosine_score}\n'
        f'typ: {r.entity_type}\n'
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

služby
skore: 1.0000001192092896
typ: skill_group

dovednosti
skore: 1.0
typ: skill_group

znalosti
skore: 1.0
typ: skill_group

řešení problémů
skore: 1.0
typ: skill_group

řízení lidských zdrojů
skore: 0.9999999403953552
typ: skill

účetnictví
skore: 0.9999998807907104
typ: skill

plány
skore: 0.9900640249252319
typ: skill

správa dodavatelů
skore: 0.9622488021850586
typ: skill

provádět nábor
skore: 0.9273862838745117
typ: skill

dodržovat předpisy
skore: 0.9263389110565186
typ: skill

management a správa
skore: 0.9194412231445312
typ: skill_group

zařídit audit
skore: 0.9039450287818909
typ: skill

učinit rozhodnutí
skore: 0.9029488563537598
typ: skill

propouštět zaměstnance
skore: 0.9027515649795532
typ: skill

péče o osoby se zdravotním postižením
skore: 0.8813368082046509
typ: skill

řídit tým
skore: 0.8810374736785889
typ: skill

strategie hráče
skore: 0.8692692518234253
typ: skill

přijímat rozhodnutí
skore: 0.8675544857978821
typ: skill_group

jednat se zúčastněnými stranami

### Pipeline 1: Analýza životopisu s cílovým povoláním

In [ ]:
# Použijte testfiles/resume.pdf, resume.docx nebo resume.odt
file_path = input("Cesta k souboru (default: testfiles/resume.pdf): ") or "testfiles/resume.pdf"
target_job = input("Cílové povolání (volitelné, např. 'programátor'): ") or None

result = documentinput_job_specific(file_path, target_job)

print(f"\nNalezeno {len(result['extracted_skills'])} dovedností a {len(result['extracted_occupations'])} povolání")
print_suggestion(result['top_suggestion'], "Nejlepší shoda")
if target_job:
    print_suggestion(result['target_suggestion'], f"Shoda pro '{target_job}'")

### Pipeline 2: Analýza životopisu → ISCO domény

In [ ]:
file_path = input("Cesta k souboru (default: testfiles/resume.pdf): ") or "testfiles/resume.pdf"

domains = documentinput_job_specific_isco(file_path)

print(f"\nNalezeno {len(domains)} ISCO domén:\n")
for domain in domains:
    print(f"{domain.domain} ({domain.occ_count} povolání)")
    for suggestion in domain.occupations[:2]:
        print(f"   • {suggestion.occupation.label} (match: {suggestion.match_score:.2%})")
    if len(domain.occupations) > 2:
        print(f"   ... a dalších {len(domain.occupations) - 2}")

### Pipeline 3: Dovednosti → Návrhy povolání

In [ ]:
skills_input = input("Zadejte dovednosti oddělené čárkou (např. 'Python, SQL, analýza dat'): ")

suggestions = textinput_match(skills_input)

print(f"\nNalezeno {len(suggestions)} návrhů povolání:\n")
for i, suggestion in enumerate(suggestions[:5], 1):
    print(f"{i}. {suggestion.occupation.label}")
    print(f"   Match score: {suggestion.match_score:.2%}")
    print(f"   ISCO: {suggestion.occupation.code}")

### Pipeline 4: Dovednosti + cílové povolání

In [ ]:
skills_input = input("Zadejte dovednosti oddělené čárkou: ")
target_job = input("Cílové povolání: ")

result = textinput_job_specific_match(skills_input, target_job)

print_suggestion(result['top_suggestion'], "Nejlepší celková shoda")
print_suggestion(result['target_suggestion'], f"Shoda pro '{target_job}'")

### Pipeline 5: Dotaz na entity

In [ ]:
query_text = input("Hledaný text (např. 'Python' nebo 'programátor'): ")
entity_type = input("Typ entity (skill/occupation/all, default: all): ") or None
n = int(input("Počet výsledků (default: 5): ") or 5)

results = text_search(query_text, entity_type, n)

print(f"\nNalezeno {len(results)} výsledků:\n")
for entity in results:
    print_entity(entity)
    print(f"      URI: {entity.esco_uri}")

### Bonus: Přímé volání query funkcí s pick_resume

In [ ]:
text = pick_resume()
print("Náhodný životopis:")
print(textwrap.shorten(text, width=400, placeholder='...'))
print("\n" + "="*50 + "\n")

entities = extract_from_resume(text)
skills = [e for e in entities if e.entity_type == 'skill']
occupations = [e for e in entities if e.entity_type == 'occupation']

print(f"Extrahované dovednosti ({len(skills)}):")
for s in skills[:5]:
    print_entity(s)

print(f"\nExtrahovaná povolání ({len(occupations)}):")
for o in occupations[:5]:
    print_entity(o)